# 预训练实操
以下实操代码参考`Chinese-LLaMA-Alpaca`中的预训练代码，代码源地址：[scripts/training/run_clm_pt_with_peft.py](https://github.com/ymcui/Chinese-LLaMA-Alpaca/blob/f213c2c53e92f2bfb41859ffdb2cf47a261c24fb/scripts/training/run_clm_pt_with_peft.py)

希望学完这个实操教程后，能够轻易的看懂原代码，如果学完看起来仍然很吃力，说明并没有完全掌握。

预训练的过程主要分为：
1. 环境准备
2. 数据准备
3. 加载Tokenizer
4. 初始化模型参数
5. 配置训练参数
6. 启动训练
7. 模型评估

## 环境配置
`pip install -r requirements.txt`

In [ ]:
import logging
import numpy as np
import math
import os
import sys
from dataclasses import dataclass, field
from itertools import chain
from typing import Optional, List, Dict, Any, Mapping
from pathlib import Path
import datasets
import torch
from datasets import load_dataset, concatenate_datasets

import transformers
from transformers import (
    CONFIG_MAPPING,
    MODEL_FOR_CAUSAL_LM_MAPPING,
    AutoConfig,
    AutoModelForCausalLM,
    LlamaForCausalLM,
    LlamaTokenizer,
    AutoTokenizer,
    HfArgumentParser,
    Trainer,
    TrainingArguments,
    is_torch_tpu_available,
    set_seed,
)
from transformers.testing_utils import CaptureLogger
from transformers.trainer_utils import get_last_checkpoint
from transformers.utils import send_example_telemetry
from transformers.utils.versions import require_version

from peft import LoraConfig, TaskType, get_peft_model, PeftModel, get_peft_model_state_dict
from transformers.trainer_utils import PREFIX_CHECKPOINT_DIR

## 配置

In [ ]:
pretrain_datasets_name = "shibing624/medical"
output_dir = "./output"
do_train = True
resume_from_checkpoint=None
tokenizer_name = ""
cache_dir = ""
use_fast_tokenizer = False
model_revision = "main"
use_auth_token = False
block_size = 1024
pretrain_dataset_dir = "./med_pretrain_txt"
pretrain_data_cache_dir = "./pretrain_data_cache"
debug_mode = False
validation_split_percentage=0.05

tokenizer_name = "/data/yuguangya/ALLYOUNEED/llama/Llama-3.2-3B-Instruct"
base_model_name = "/data/yuguangya/ALLYOUNEED/llama/Llama-3.2-3B-Instruct"
model_name = "/data/yuguangya/ALLYOUNEED/llama/Llama-3.2-3B-Instruct"

## 构造Tokenizer

In [ ]:
tokenizer_kwargs = {
    # "cache_dir": cache_dir,
    "use_fast": use_fast_tokenizer,
    # "revision": model_revision,
    # "use_auth_token": True if use_auth_token else None,
}
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, **tokenizer_kwargs)## 构造Tokenizer

In [ ]:
print(tokenizer)
print(tokenizer.eos_token)
print(tokenizer.pad_token)

## 数据集准备
训练数据集使用的是[`shibing624/medical`](https://huggingface.co/datasets/shibing624/medical)，这是一个用于医疗领域预训练的医疗数据集，数据集介绍如下：
1. pretrain
- train_encyclopedia.json: 共36万条，来自医疗百科数据FreedomIntelligence/huatuo_encyclopedia_qa , 拼接 questions 和 answers，形成 text 文本字段，语句通顺，用于预训练注入医疗知识。
- medical_book_zh.json: 共8475条，来自医疗教材的文本数据，来源：https://github.com/jind11/MedQA， 原始数据集：google drive ，只对长段落切分为2048字的小段落了。
2. finetune
- train_zh_0.json: 共195万条，来自1）中文医疗对话数据集Toyhom/Chinese-medical-dialogue-data的六个科室医疗问诊数据， 有79万条；2）在线医疗百科 huatuo_encyclopedia_qa ，有36万条；3）医疗知识图谱 huatuo_knowledge_graph_qa，有79万条。三部分合并，共195万条。
- train_en_1.json：共11万条，来自英文医疗问诊对话数据Kent0n-Li/ChatDoctor，合并了HealthCareMagic-100k、GenMedGPT-5k 数据集，共11万条。
3. reward
- train.json 共4000条，问题来自中文医疗对话数据集Toyhom/Chinese-medical-dialogue-data的随机4000条提问，response_chosen来自该数据集的医生答复， response_rejected来自本草模型SCIR-HI/Huatuo-Llama-Med-Chinese的答复。

### 加载数据集
预训练数据集使用的是从丁香医生网站爬取的医疗文本，可以运行[Crawl_DX](../_数据爬取/Crawl_DX.py)自动爬取，爬取完成的数据保存到[dataset_txt](./med_pretrain_txt/)中，下面加载训练数据集的代码会遍历这个目录下的全部txt文件，读取并预处理。


In [ ]:
from pathlib import Path
from itertools import chain
import os
from transformers.testing_utils import CaptureLogger
import logging
logger = logging.getLogger(__name__)

tok_logger = transformers.utils.logging.get_logger("transformers.tokenization_utils_base")

def tokenize_function(examples):
    with CaptureLogger(tok_logger) as cl:
        output = tokenizer(examples["text"])
    # clm input could be much much longer than block_size
    if "Token indices sequence length is longer than the" in cl.out:
        tok_logger.warning(
            "^^^^^^^^^^^^^^^^ Please ignore the warning above - this long input will be chunked into smaller bits"
            " before being passed to the model."
        )
    return output

def group_texts(examples):
        # Concatenate all texts.
    concatenated_examples = {k: list(chain(*examples[k])) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of max_len.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

# 如果使用与训练的
lm_datasets = []
# lm_datasets=DatasetDict()
path = Path(pretrain_dataset_dir)
files = [file.name for file in path.glob("*.txt")]
if debug_mode is True:
    files = [files[0]]
for idx, file in enumerate(files):
    data_file = os.path.join(path, file)
    filename = ''.join(file.split(".")[:-1])
    cache_path = os.path.join(pretrain_data_cache_dir, filename)
    os.makedirs(cache_path, exist_ok=True)
    try:
        processed_dataset = datasets.load_from_disk(cache_path, keep_in_memory=False)
        print(f'training datasets-{filename} has been loaded from disk')
        
    except Exception as e:
        cache_dir = os.path.join(pretrain_data_cache_dir, filename+"_text")
        os.makedirs(cache_dir, exist_ok=True)
        raw_dataset = load_dataset("text", data_files=data_file, cache_dir=cache_dir, keep_in_memory=False)
        print(f"{file} has been loaded")
        tokenized_dataset = raw_dataset.map(
            tokenize_function,
            batched=True,
            num_proc=8,
            remove_columns="text",
            load_from_cache_file=True,
            keep_in_memory=False,
            cache_file_names = {k: os.path.join(cache_dir, 'tokenized.arrow') for k in raw_dataset},
            desc="Running tokenizer on dataset",
        )
        grouped_datasets = tokenized_dataset.map(
            group_texts,
            batched=True,
            num_proc=8,
            load_from_cache_file=True,
            keep_in_memory=False,
            cache_file_names = {k: os.path.join(cache_dir, 'grouped.arrow') for k in tokenized_dataset},
            desc=f"Grouping texts in chunks of {block_size}",
        )
        processed_dataset = grouped_datasets
        processed_dataset.save_to_disk(cache_path)
    if idx == 0:
        lm_datasets = processed_dataset['train']
    else:
        assert lm_datasets.features.type == processed_dataset["train"].features.type
        lm_datasets = concatenate_datasets([lm_datasets, processed_dataset["train"]])

lm_datasets = lm_datasets.train_test_split(test_size = validation_split_percentage)

print(tokenizer.decode(lm_datasets['train'][10]['input_ids']))
print(tokenizer.decode(lm_datasets['test'][10]['input_ids']))

In [ ]:
print(lm_datasets)

In [ ]:
train_dataset = lm_datasets['train']
print(f"Num train_samples  {len(train_dataset)}")
print("training example:")
print(tokenizer.decode(train_dataset[0]['input_ids']))
eval_dataset = lm_datasets["test"]
print(f"Num eval_samples  {len(eval_dataset)}")
print("training example:")
print(tokenizer.decode(eval_dataset[0]['input_ids']))

In [ ]:
# 该数据集已经包含Pretrained、fintune、Reward数据集代码， 仅加载Reward，用于教程
# Pretrained采用加载txt的方式，通用性更好
datasets = load_dataset(pretrain_datasets_name, 'reward')

## 加载预训练模型
加载的模型为3B的LLaMA3.2模型

In [ ]:
torch_dtype = 'float16'
model_name_or_path = "/data/yuguangya/ALLYOUNEED/llama/Llama-3.2-3B-Instruct"
cache_dir = "./cache"
torch_dtype = (
    torch_dtype
    if torch_dtype in ["auto", None]
    else getattr(torch, torch_dtype)
)

config_kwargs = {
    "cache_dir": cache_dir,
    "revision": model_revision,
    "use_auth_token": True if use_auth_token else None,
}

config = AutoConfig.from_pretrained(model_name_or_path, **config_kwargs)

model = LlamaForCausalLM.from_pretrained(
    model_name_or_path,
    from_tf=bool(".ckpt" in model_name_or_path),
    config=config,
    cache_dir=cache_dir,
    revision=model_revision,
    use_auth_token=True if use_auth_token else None,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True
)

In [ ]:
model_vocab_size = model.get_output_embeddings().weight.size(0)
model.resize_token_embeddings(len(tokenizer))

## 加载LoRA配置


In [ ]:
print("Init new peft model")
lora_rank=8
lora_alpha=32
lora_trainable="q_proj,v_proj,k_proj,o_proj,gate_proj,down_proj,up_proj"
modules_to_save="embed_tokens,lm_head"
lora_dropout=0.05

target_modules = lora_trainable.split(',')
modules_to_save = modules_to_save
if modules_to_save is not None:
    modules_to_save = modules_to_save.split(',')
lora_rank = lora_rank
lora_dropout = lora_dropout
lora_alpha = lora_alpha
print(f"target_modules: {target_modules}")
print(f"lora_rank: {lora_rank}")
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=target_modules,
    inference_mode=False,
    r=lora_rank, lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    modules_to_save=modules_to_save)
model = get_peft_model(model, peft_config)

In [ ]:

model.print_trainable_parameters()
old_state_dict = model.state_dict
model.state_dict = (
    lambda self, *_, **__: get_peft_model_state_dict(self, old_state_dict())
).__get__(model, type(model))

## 初始化Trainer


In [ ]:
# max_steps = 10
eval_freq = 500
save_freq = 500
log_freq = 10
num_train_epochs = 1
model_pretrained_name = ""
do_eval=True

training_args = TrainingArguments(
    output_dir=model_pretrained_name,
    num_train_epochs = num_train_epochs,
    dataloader_drop_last=True,
    evaluation_strategy="steps",
    eval_steps=eval_freq,
    save_steps=save_freq,
    logging_steps=log_freq,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=16,
    # max_steps=max_steps,
    warmup_steps=100,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    lr_scheduler_type="cosine",
    weight_decay=0.05,
    fp16=False,
    logging_first_step=True,
    # report_to="wandb",
    max_steps=10, # 为了调试方便，设置为10步
)

from sklearn.metrics import accuracy_score
def accuracy(predictions, references, normalize=True, sample_weight=None):
    return {
        "accuracy": float(
            accuracy_score(references, predictions, normalize=normalize, sample_weight=sample_weight)
        )
    }

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    # preds have the same shape as the labels, after the argmax(-1) has been calculated
    # by preprocess_logits_for_metrics but we need to shift the labels
    labels = labels[:, 1:].reshape(-1)
    preds = preds[:, :-1].reshape(-1)
    return accuracy(predictions=preds, references=labels)

from transformers import AutoTokenizer, AutoConfig, DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        # Depending on the model and config, logits may contain extra tensors,
        # like past_key_values, but logits always come first
        logits = logits[0]
    return logits.argmax(dim=-1)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset if training_args.do_train else None,
    eval_dataset=eval_dataset if training_args.do_eval else None,
    tokenizer=tokenizer,
    data_collator=data_collator, #fault_tolerance_data_collator,
    compute_metrics=compute_metrics if do_eval and not is_torch_tpu_available() else None,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics
    if training_args.do_eval and not is_torch_tpu_available()
    else None,
)

## 开始训练


In [ ]:
max_train_samples = 1000

train_result = trainer.train()

metrics = train_result.metrics

max_train_samples = (
    max_train_samples if max_train_samples is not None else len(train_dataset)
)
metrics["train_samples"] = min(max_train_samples, len(train_dataset))

trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
trainer.save_state()

## 评估


In [ ]:
max_eval_samples = 10
metrics = trainer.evaluate()

max_eval_samples = max_eval_samples if max_eval_samples is not None else len(eval_dataset)
metrics["eval_samples"] = min(max_eval_samples, len(eval_dataset))
try:
    perplexity = math.exp(metrics["eval_loss"])
except OverflowError:
    perplexity = float("inf")
metrics["perplexity"] = perplexity

trainer.log_metrics("eval", metrics)
trainer.save_metrics("eval", metrics)